1. Veri setini yükleme
2. MLP yapısını tanımlama
3. Model parametrelerini başlatma
4. İleri yayılım (forward propagation)
5. Maliyet (cost) hesaplama 
6. Geri yayılım (Backpropagation)
7. Parametre güncelleme Update Propagation
8. Tüm adımların entegrasyonu

## Veri setini yükleme

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("BankNote_Authentication.csv")

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1372 entries, 0 to 1371
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   variance  1372 non-null   float64
 1   skewness  1372 non-null   float64
 2   curtosis  1372 non-null   float64
 3   entropy   1372 non-null   float64
 4   class     1372 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 53.7 KB


In [4]:
df.head(50)

,variance,skewness,curtosis,entropy,class
0,3.621600,8.66610,-2.80730,-0.446990,0
1,4.545900,8.16740,-2.45860,-1.462100,0
2,3.866000,-2.63830,1.92420,0.106450,0
3,3.456600,9.52280,-4.01120,-3.594400,0
4,0.329240,-4.45520,4.57180,-0.988800,0
5,4.368400,9.67180,-3.96060,-3.162500,0
6,3.591200,3.01290,0.72888,0.564210,0
7,2.092200,-6.81000,8.46360,-0.602160,0
8,3.203200,5.75880,-0.75345,-0.612510,0
9,1.535600,9.17720,-2.27180,-0.735350,0


In [5]:
df.describe()

,variance,skewness,curtosis,entropy,class
count,1372.000000,1372.000000,1372.000000,1372.000000,1372.000000
mean,0.433735,1.922353,1.397627,-1.191657,0.444606
std,2.842763,5.869047,4.310030,2.101013,0.497103
min,-7.042100,-13.773100,-5.286100,-8.548200,0.000000
25%,-1.773000,-1.708200,-1.574975,-2.413450,0.000000
50%,0.496180,2.319650,0.616630,-0.586650,0.000000
75%,2.821475,6.814625,3.179250,0.394810,1.000000
max,6.824800,12.951600,17.927400,2.449500,1.000000


In [6]:
df.isnull().sum()

variance    0
skewness    0
curtosis    0
entropy     0
class       0
dtype: int64

In [7]:
df = df.sample(frac = 1, random_state = 42).reset_index(drop= True)

In [8]:
df.head(50)

,variance,skewness,curtosis,entropy,class
0,1.569100,6.346500,-0.18280,-2.409900,0
1,-0.278020,8.188100,-3.13380,-2.527600,0
2,0.051979,7.052100,-2.05410,-3.150800,0
3,-1.755900,11.945900,3.09460,-4.897800,0
4,2.428700,9.382100,-3.24770,-1.454300,0
5,4.635200,-3.008700,2.67730,1.212000,0
6,0.248350,7.643900,0.98850,-0.873710,0
7,5.173100,3.960600,-1.98300,0.407740,0
8,4.004700,0.459370,1.36210,1.618100,0
9,0.964410,5.839500,2.32350,0.066365,0


In [25]:
X, y = df.iloc[:, :-1], df.iloc[:, -1]

In [26]:
print("X type is " + str(type(X)))
print("X shape is " + str(X.shape))
print("y type is " + str(type(y)))
print("y shape is " + str(y.shape))

X type is <class 'pandas.core.frame.DataFrame'>
X shape is (1372, 4)
y type is <class 'pandas.core.series.Series'>
y shape is (1372,)


In [27]:
X = X.to_numpy()

In [28]:
y = y.to_numpy().reshape(-1, 1)

In [29]:
print("X type is " + str(type(X)))
print("X shape is " + str(X.shape))
print("y type is " + str(type(y)))
print("y shape is " + str(y.shape))

X type is <class 'numpy.ndarray'>
X shape is (1372, 4)
y type is <class 'numpy.ndarray'>
y shape is (1372, 1)


In [30]:
from sklearn.model_selection import train_test_split

In [31]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)

In [32]:
import numpy as np

print("Train class distribution:")
unique, counts = np.unique(y_train, return_counts=True)
print(dict(zip(unique, counts)))

print("Test class distribution:")
unique, counts = np.unique(y_test, return_counts=True)
print(dict(zip(unique, counts)))

Train class distribution:
{np.int64(0): np.int64(609), np.int64(1): np.int64(488)}
Test class distribution:
{np.int64(0): np.int64(153), np.int64(1): np.int64(122)}


In [33]:
print("X train shape is " + str(X_train.shape))
print("X test shape is " + str(X_test.shape))
print("y train shape is " + str(y_train.shape))
print("y test shape is " + str(y_test.shape))

X train shape is (1097, 4)
X test shape is (275, 4)
y train shape is (1097, 1)
y test shape is (275, 1)


## Model Mimarisi (Define Network Structure)

Input layer size is X.shape[1]
Hidden layer size is 5.
Output layer size is 1.

Örneğin; giriş katmanı boyutu X.shape[1], gizli katman nöron sayısı 5 ve çıkış katmanı 1 olabilir. 

In [34]:
print("X.shape is " + str(X.shape))
print("X.shape[1] is " + str(X.shape[1]))

X.shape is (1372, 4)
X.shape[1] is 4


## Modeli Başlatmak (Initialize Model Parameters)

In [35]:
def initialize_parameters(n_x, n_h, n_y = 1):
    np.random.seed(42)
    W1 = np.random.randn(n_h, n_x) * 0.01
    b1 = np.zeros((n_h, 1))
    W2 = np.random.randn(n_y, n_h) * 0.01
    b2 = np.zeros((n_y, 1))
    
    parameters = {
        "W1" : W1,
        "b1" : b1,
        "W2" : W2,
        "b2" : b2
    }
    return parameters

In [36]:
test_parameters = initialize_parameters(X.shape[1], 5, 1)

In [37]:
print("W1 = " + str(test_parameters["W1"]))
print("b1 = " + str(test_parameters["b1"]))
print("W2 = " + str(test_parameters["W2"]))
print("b2 = " + str(test_parameters["b2"]))

W1 = [[ 0.00496714 -0.00138264  0.00647689  0.0152303 ]
 [-0.00234153 -0.00234137  0.01579213  0.00767435]
 [-0.00469474  0.0054256  -0.00463418 -0.0046573 ]
 [ 0.00241962 -0.0191328  -0.01724918 -0.00562288]
 [-0.01012831  0.00314247 -0.00908024 -0.01412304]]
b1 = [[0.]
 [0.]
 [0.]
 [0.]
 [0.]]
W2 = [[ 0.01465649 -0.00225776  0.00067528 -0.01424748 -0.00544383]]
b2 = [[0.]]


In [38]:
print("W1 = " + str(test_parameters["W1"].shape))
print("b1 = " + str(test_parameters["b1"].shape))
print("W2 = " + str(test_parameters["W2"].shape))
print("b2 = " + str(test_parameters["b2"].shape))
print("X_train = " + str(X_train.shape))

W1 = (5, 4)
b1 = (5, 1)
W2 = (1, 5)
b2 = (1, 1)
X_train = (1097, 4)


## Forward Propagation

In [39]:
def forward_propagation(X, parameters):
    W1 = parameters['W1']
    b1 = parameters['b1']
    W2 = parameters['W2']
    b2 = parameters['b2']

    Z1 = np.dot(W1, X.T) + b1

    A1 = np.tanh(Z1)

    Z2 = np.dot(W2, A1) + b2

    A2 = sigmoid(Z2)

    cache = {
        "Z1": Z1,
        "A1": A1,
        "Z2": Z2,
        "A2": A2
    }

    return A2, cache

In [40]:
def sigmoid(Z):
    return 1 / (1 + np.exp(-1 * Z))

In [41]:
test_A2, test_cache = forward_propagation(X_train, test_parameters)

In [42]:
print("A2 test:" + str(test_A2))
print("A2 test shape:" + str(test_A2.shape))

A2 test:[[0.50004348 0.5003625  0.50046776 ... 0.50019115 0.50022992 0.49971893]]
A2 test shape:(1, 1097)


In [43]:
print("Z1 is:" + str(test_cache["Z1"]))
print("A1 is:" + str(test_cache["A1"]))
print("Z2 is:" + str(test_cache["Z2"]))
print("A2 is:" + str(test_cache["A2"]))

Z1 is:[[-0.102913    0.05031256  0.02529391 ...  0.06187844  0.00323155
  -0.05184445]
 [-0.03184479  0.21452315  0.0202957  ...  0.1732279  -0.00842498
  -0.08684872]
 [ 0.09539668 -0.07726144 -0.00443225 ... -0.08418797  0.00827182
   0.04041363]
 [-0.16539789 -0.05743321 -0.09467096 ...  0.0139666  -0.05682372
   0.01772067]
 [ 0.14697393 -0.07822253 -0.03756487 ... -0.09243179 -0.00716118
   0.06171007]]
A1 is:[[-0.10255121  0.05027015  0.02528852 ...  0.06179958  0.00323153
  -0.05179805]
 [-0.03183403  0.21129183  0.02029291 ...  0.17151571 -0.00842478
  -0.08663102]
 [ 0.09510835 -0.07710808 -0.00443222 ... -0.08398964  0.00827163
   0.04039165]
 [-0.16390598 -0.05737015 -0.09438914 ...  0.01396569 -0.05676263
   0.01771882]
 [ 0.14592472 -0.07806337 -0.03754721 ... -0.09216945 -0.00716106
   0.06163186]]
Z2 is:[[ 0.00017392  0.00145001  0.00187104 ...  0.00076458  0.00091968
  -0.00112427]]
A2 is:[[0.50004348 0.5003625  0.50046776 ... 0.50019115 0.50022992 0.49971893]]


## Compute Cost

In [44]:
print("Y shape is " +str(y_train.shape))

Y shape is (1097, 1)


In [45]:
def compute_cost(A2, Y):
    m = A2.shape[1]
    cost = - (np.dot(np.log(A2), Y) + np.dot(np.log(1 - A2), (1 - Y))) / m
    cost = float(np.squeeze(cost))
    return cost

In [46]:
test_cost = compute_cost(test_A2, y_train) 

In [47]:
print("test_cost is " + str(test_cost))
print("test_cost type is " + str(type(test_cost)))

test_cost is 0.6934831763208148
test_cost type is <class 'float'>


## Backpropagation

In [48]:
print("A1 shape" + str(test_cache["A1"].shape))
print("A2 Shape" + str(test_A2.shape))
print("A2 Shape" + str(test_cache["A2"].shape))
print("W1 shape" + str(test_parameters["W1"].shape))
print("W2 shape" + str(test_parameters["W2"].shape))
print("y train shape" + str(y_train.shape))
print("X_train shape " + str(X_train.shape))

A1 shape(5, 1097)
A2 Shape(1, 1097)
A2 Shape(1, 1097)
W1 shape(5, 4)
W2 shape(1, 5)
y train shape(1097, 1)
X_train shape (1097, 4)


In [53]:
def backpropagation(X, Y, cache, parameters):
    m = X.shape[0]

    W2 = parameters['W2']
    A1 = cache['A1']
    A2 = cache['A2']

    Y = Y.reshape(A2.shape)

    dZ2 = A2 - Y

    dW2 = (1/m) * np.dot(dZ2, A1.T)

    db2 = (1/m) * np.sum(dZ2, axis=1, keepdims=True)

    dZ1 = np.dot(W2.T, dZ2) * (1 - np.power(A1, 2))

    dW1 = (1/m) *np.dot(dZ1, X)

    db1 = (1/m) * np.sum(dZ1, axis=1, keepdims=True)

    grads = {
        "dW1": dW1,
        "db1": db1,
        "dW2": dW2,
        "db2": db2
    }
    
    return grads  

In [54]:
test_grads = backpropagation(X_train, y_train, test_cache, test_parameters)

In [55]:
print("dW1 shape is" + str(test_grads["dW1"].shape))
print("dW2 shape is" + str(test_grads["dW2"].shape))
print("db1 shape is" + str(test_grads["db1"].shape))
print("db2 shape is" + str(test_grads["db2"].shape))
print("b1 shape is" + str(test_parameters["b1"].shape))

dW1 shape is(5, 4)
dW2 shape is(1, 5)
db1 shape is(5, 1)
db2 shape is(1, 1)
b1 shape is(5, 1)


In [66]:
def update_parameters(parameters, grads, learning_rate = 0.01):
    W1 = parameters["W1"]
    b1 = parameters["b1"]
    W2 = parameters["W2"]
    b2 = parameters["b2"]
    
    dW1 = grads["dW1"]
    db1 = grads["db1"]
    dW2 = grads["dW2"]
    db2 = grads["db2"]
    
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2
    
    paramaters = {
        "W1" : W1,
        "b1" : b1,
        "W2" : W2,
        "b2" : b2
    }
    
    return paramaters

In [67]:
test_learned_parameters = update_parameters(test_parameters, test_grads)

In [68]:
test_learned_parameters["b2"].shape

(1, 1)

In [69]:
def nn_model(X, Y, n_x, n_h, n_y, n_steps = 100, print_cost = True):
    parameters = initialize_parameters(n_x, n_h, n_y)
    
    for i in range(0, n_steps):
        A2, cache = forward_propagation(X, parameters)
        cost = compute_cost(A2, Y)
        grads = backpropagation(X, Y, cache, parameters)
        parameters = update_parameters(parameters, grads)
        
        if print_cost and i % 10 == 0:
            print("cost %i %f" %(i,cost))
    
    return parameters
        

In [70]:
X_train.shape

(1097, 4)

In [71]:
parameters = nn_model(X_train, y_train, X_train.shape[1], n_h = 6, n_y = 1, n_steps = 500)

cost 0 0.693000
cost 10 0.692562
cost 20 0.692107
cost 30 0.691613
cost 40 0.691055
cost 50 0.690401
cost 60 0.689607
cost 70 0.688618
cost 80 0.687365
cost 90 0.685762
cost 100 0.683709
cost 110 0.681094
cost 120 0.677802
cost 130 0.673725
cost 140 0.668770
cost 150 0.662870
cost 160 0.655986
cost 170 0.648103
cost 180 0.639223
cost 190 0.629361
cost 200 0.618531
cost 210 0.606748
cost 220 0.594025
cost 230 0.580384
cost 240 0.565866
cost 250 0.550541
cost 260 0.534520
cost 270 0.517947
cost 280 0.500998
cost 290 0.483859
cost 300 0.466716
cost 310 0.449737
cost 320 0.433071
cost 330 0.416837
cost 340 0.401129
cost 350 0.386015
cost 360 0.371539
cost 370 0.357728
cost 380 0.344590
cost 390 0.332123
cost 400 0.320314
cost 410 0.309143
cost 420 0.298585
cost 430 0.288612
cost 440 0.279193
cost 450 0.270298
cost 460 0.261896
cost 470 0.253955
cost 480 0.246448
cost 490 0.239345


In [72]:
def predict(parameters, X):
    A2, cache = forward_propagation(X, parameters)
    predicts = A2 > 0.5
    return predicts

In [73]:
predicts = predict(parameters, X_test)

In [74]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
y_true = y_test.flatten()  
y_pred = predicts.flatten() 
acc = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='binary') 
recall = recall_score(y_true, y_pred, average='binary')
f1 = f1_score(y_true, y_pred, average='binary')
conf_matrix = confusion_matrix(y_true, y_pred)
print(f"Accuracy: {acc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print("\nConfusion Matrix:")
print(conf_matrix)
print("\nClassification Report:")
print(classification_report(y_true, y_pred)) 

Accuracy: 0.9564
Precision: 1.0000
Recall: 0.9016
F1 Score: 0.9483

Confusion Matrix:
[[153   0]
 [ 12 110]]

Classification Report:
              precision    recall  f1-score   support

           0       0.93      1.00      0.96       153
           1       1.00      0.90      0.95       122

    accuracy                           0.96       275
   macro avg       0.96      0.95      0.96       275
weighted avg       0.96      0.96      0.96       275



In [75]:
parameters_n_h = [i for i in range(3, 11)]
parameters_n_steps = [i for i in range(100, 1100, 100)]
for n_h in parameters_n_h:
    for n_step  in parameters_n_steps:
        parameters = nn_model(X_train, y_train, X_train.shape[1], n_h = n_h, n_y = 1, n_steps = n_step, print_cost = False)
        predicts = predict(parameters, X_test)
        acc = accuracy_score(y_test.flatten(), predicts.flatten())
        print(f"n_h: {n_h}, n_step: {n_step}, acc: {acc}")

n_h: 3, n_step: 100, acc: 0.5563636363636364
n_h: 3, n_step: 200, acc: 0.7490909090909091
n_h: 3, n_step: 300, acc: 0.8581818181818182
n_h: 3, n_step: 400, acc: 0.9345454545454546
n_h: 3, n_step: 500, acc: 0.9563636363636364
n_h: 3, n_step: 600, acc: 0.9709090909090909
n_h: 3, n_step: 700, acc: 0.9745454545454545
n_h: 3, n_step: 800, acc: 0.9818181818181818
n_h: 3, n_step: 900, acc: 0.9818181818181818
n_h: 3, n_step: 1000, acc: 0.9818181818181818
n_h: 4, n_step: 100, acc: 0.5563636363636364
n_h: 4, n_step: 200, acc: 0.7236363636363636
n_h: 4, n_step: 300, acc: 0.8472727272727273
n_h: 4, n_step: 400, acc: 0.9309090909090909
n_h: 4, n_step: 500, acc: 0.9563636363636364
n_h: 4, n_step: 600, acc: 0.9709090909090909
n_h: 4, n_step: 700, acc: 0.9745454545454545
n_h: 4, n_step: 800, acc: 0.9781818181818182
n_h: 4, n_step: 900, acc: 0.9818181818181818
n_h: 4, n_step: 1000, acc: 0.9818181818181818
n_h: 5, n_step: 100, acc: 0.5563636363636364
n_h: 5, n_step: 200, acc: 0.7090909090909091
n_h: 5, 